# Final Merchant Payable Reconciliation Output

Notebook-friendly pandas view of the dummy-data reconciliation output.

In [ ]:
import importlib
import subprocess
import sys
from pathlib import Path

for package in ["duckdb", "pandas"]:
    try:
        importlib.import_module(package)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import duckdb
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

In [ ]:
candidate_paths = [
    Path.cwd() / "merchant_settlement_reconciliation.duckdb",
    Path.cwd().parent / "merchant_settlement_reconciliation.duckdb",
]
db_path = next((path for path in candidate_paths if path.exists()), None)

if db_path is None:
    raise FileNotFoundError(
        "Could not find merchant_settlement_reconciliation.duckdb. "
        "Run this notebook from the repo root or the notebooks folder after running dbt build."
    )

query = """
select
    posting_date,
    legal_entity,
    currency,
    settlement_net,
    gl_net,
    variance,
    reconciliation_status,
    investigation_priority,
    deterministic_root_cause_hint,
    evidence_hash
from main_subledger.audit_merchant_payable_tieout
order by posting_date, legal_entity, currency
"""

with duckdb.connect(str(db_path), read_only=True) as con:
    final_df = con.execute(query).df()

final_df

## Screenshot View

In [ ]:
snippet_df = final_df[
    [
        "posting_date",
        "legal_entity",
        "currency",
        "settlement_net",
        "gl_net",
        "variance",
        "reconciliation_status",
        "investigation_priority",
    ]
].copy()

def highlight_status(row):
    status = row["reconciliation_status"]
    if status == "MATCHED":
        return ["background-color: #e8f5e9"] * len(row)
    if status == "AMOUNT_VARIANCE":
        return ["background-color: #fff3cd"] * len(row)
    return ["background-color: #fdecea"] * len(row)

styled_snippet = (
    snippet_df.style
    .format(
        {
            "settlement_net": "{:,.2f}",
            "gl_net": "{:,.2f}",
            "variance": "{:,.2f}",
        }
    )
    .apply(highlight_status, axis=1)
    .hide(axis="index")
)

display(styled_snippet)

In [ ]:
summary_df = (
    final_df.groupby(["reconciliation_status", "investigation_priority"], as_index=False)
    .size()
    .rename(columns={"size": "row_count"})
)

summary_df